# Library energy vs Log10GFP

Per-library joint distribution of **element model energy** (x) against **measured Log10GFP** (y),
one independent figure per library, with a marginal histogram on each axis and a fitted
**Boltzmann curve** through the scatter.

The aim is to *look at the distribution*. Unlike Batch 0.5 of
`Model_CorePromoter_recursive_design.ipynb` — which exists to pick `ENERGY_BIN_RANGES` and is
therefore covered in bin-edge lines, excluded-range shading and an `energy_fraction` axis — nothing
here is overlaid with design binning. All rows are plotted unfiltered: several libraries saturate at
their LogGFP limits and those pile-ups are real information about the assay.

| Key | Table | Points | Element | Score source |
|---|---|---|---|---|
| `UL` | UL.pkl | 177,582 | UP | weights_UP.pt |
| `SL17` | SL17.pkl | 333,565 | spacer | weights_Sp17.pt |
| `DL` | DL.pkl | 54,706 | DIS | weights_Dis.pt |
| `ITS` | ITS.pkl | 945,398 | ITS | weights_ITS.pt |
| `PL_m35` | PL.pkl | 2,584 | -35 | BPM `-dG` |
| `PL_m10` | PL.pkl | 1,330 | -10 | BPM `-dG` |
| `PL_combined` | PL.pkl | 1,022,018 | -35 + spacer + -10 | BPM summed `-dG` |

SL16 and SL18 are not in the default set — use the ad-hoc cell at the bottom to plot them, or any
other table, without touching the rest of the notebook.

## How PL is handled

A PL row binds a `minus35` **and** a `minus10` (plus a varying `spacer_length`) to a single
`LogGFP`, so no *individual* element's energy explains that row on its own. There are two ways round
this, and the notebook does both.

**Per 6-mer (`PL_m35`, `PL_m10`).** Each distinct 6-mer gets one point, whose Log10GFP is the mean
of every PL row containing it, computed independently for the -35 and the -10 column. Support is
extremely uneven — `minus35` ranges from 1 to 461,858 rows (median 25), `minus10` from 1 to 193,632
(median 19) — so a 6-mer seen once sits beside one averaged over hundreds of thousands.
`PL_MIN_ROWS` keeps every 6-mer by default; raising it sharpens the relationship considerably (for
-35, Boltzmann R2 goes 0.07 → 0.14 → 0.21 → 0.38 at thresholds 1 / 5 / 20 / 100).

**Summed promoter (`PL_combined`).** Summing all three core elements removes the aggregation
entirely: each row has one combined energy and its own LogGFP, so **nothing is averaged** and all
1,022,018 rows are real points. Energy is `BPM.score_promoter` split into its additive dG terms and
negated: `energy = -(dG_m35 + dG_spacer + dG_m10)`, with the spacer term controlled by
`PL_INCLUDE_SPACER`.

| x axis | Pearson r | Boltzmann R2 |
|---|---|---|
| -35 only | 0.128 | 0.074 |
| -10 only | 0.216 | 0.072 |
| -35 + -10 | 0.273 | 0.129 |
| **-35 + spacer + -10** | **0.315** | **0.190** |

Two caveats on `PL_combined`: the fitted `e0` (10.39) falls **outside** the observed energy range
(max 8.39) and 99% of rows sit below 6.10, so the right-hand end of the curve is extrapolation and
`e0` should not be read as a physical quantity. And 10.7% of rows (109,382) pile up at the LogGFP
floor of 0.3394, which is why Spearman rho (0.150) is so far below Pearson r (0.315); excluding that
population only moves r to 0.329, so the modest correlation reflects the limits of the additive BPM
energy rather than floor noise.

## The Boltzmann curve

Both scoring paths convert energy to expression through the same two-state occupancy with **beta
fixed at 1/(R*T)**, R = 0.001987 kcal/mol/K and T = 310 K:

- `ElementEnergyModel.energy2expression` (`recursive_corepromoter_design.py`) uses
  `exp(+beta*(e + e0))`, since higher energy is stronger;
- `BPM.score2exp` (`BPM/BPM.py`) uses `exp(-beta*(dG + cE0))`, since lower dG is stronger.

The x axis here is already higher-is-stronger (`ElementModelBundle.score` negates the BPM dG), so a
single form covers both:

```text
GFP(e) = GFP_min + (GFP_max - GFP_min) * sigmoid(beta * (e - e0))
```

This looks unlike `BPM.score2exp` but is the same equation. BPM writes the ratio
`(cmin + cmax*B)/(1+B)`; substituting `s = B/(1+B)` (so `1-s = 1/(1+B)`) turns it into
`cmin*(1-s) + cmax*s = cmin + (cmax-cmin)*s`, and with `e = -dG` the occupancy becomes
`s = sigmoid(beta*(e - cE0))`. So `GFP_min = cmin`, `GFP_max = cmax`, `e0 = cE0`. Verified against
`BPM.score2exp` numerically — the two agree to ~1e-13.

Beware of reading a fitted `e0` as if it were BPM's `cE0 = 7.30`: BPM's constants are calibrated on
the **whole promoter** dG, whereas the six element panels each fit a single element, whose upper
plateau therefore sits far below `cmax = 1000`.

Only `e0` and the two plateaus are fitted — **beta is not a free parameter**. Fitted plateaus are
bounded to stay near the observed expression range, otherwise a library that never reaches its own
lower asymptote (ITS) drives `GFP_min` to ~1e-6 with no improvement in fit.

Outputs: `../figures/EnergyVsLogGFP_<KEY>.{png,svg}`.


In [ ]:
# === Setup and library registry ===
import importlib
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from scipy.optimize import curve_fit
from scipy.special import expit

import recursive_corepromoter_design as legacy
import automated_promoter_library_design as r

importlib.reload(legacy)
importlib.reload(r)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FIG_DIR = legacy.MS2_DIR / "figures"
CACHE_PATH = legacy.MS2_DIR / "outputs" / "library_energy_vs_loggfp.pkl"
FIG_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

# Set False to force a full rescore even when the cached signature still matches.
USE_CACHE = True

# Point style, applied to every panel. Raise POINT_ALPHA_GAIN for more opaque points;
# the floor and ceiling keep a 1.3k-point panel and a 1M-point panel both readable.
# Resulting alpha: PL_m10/PL_m35 0.85, DL 0.219, everything denser 0.09.
POINT_SIZE_GAIN = 1500.0
POINT_SIZE_RANGE = (3.0, 26.0)
POINT_ALPHA_GAIN = 12000.0
POINT_ALPHA_RANGE = (0.09, 0.85)

# Minimum number of PL rows a 6-mer must appear in to be kept. 1 keeps everything,
# which is the honest default; the single-row 6-mers are pure noise, so raising this
# sharpens the -35/-10 panels a lot (Boltzmann R2 for -35: 0.07 / 0.14 / 0.21 / 0.38
# at thresholds 1 / 5 / 20 / 100). Only affects PL_m35 and PL_m10.
PL_MIN_ROWS = 1

# Whether PL_combined includes the spacer-length dG term, as BPM.score_promoter does.
# False drops it (Pearson r 0.315 -> 0.273).
PL_INCLUDE_SPACER = True

# source="index"    -> the pkl index is the sequence, LogGFP is taken per row.
# source=<column>   -> group PL rows by that column and average LogGFP per 6-mer,
#                      because one PL row binds a minus35 AND a minus10 to one LogGFP.
# source="combined" -> per-row whole-promoter BPM energy; nothing aggregated.
LIBRARIES = {
    "UL":     dict(file="UL.pkl",   element="UP",     source="index",   label="UP element (UL, 19 bp)"),
    "SL17":   dict(file="SL17.pkl", element="spacer", source="index",   label="Spacer 17 bp (SL17)"),
    "DL":     dict(file="DL.pkl",   element="DIS",    source="index",   label="DIS element (DL, 8 bp)"),
    "ITS":    dict(file="ITS.pkl",  element="ITS",    source="index",   label="ITS element (10 bp)"),
    "PL_m35": dict(file="PL.pkl",   element="m35",    source="minus35", label="-35 element (PL, 6 bp, mean per 6-mer)"),
    "PL_m10": dict(file="PL.pkl",   element="m10",    source="minus10", label="-10 element (PL, 6 bp, mean per 6-mer)"),
    "PL_combined": dict(file="PL.pkl", element="promoter", source="combined",
                        label="-35 + spacer + -10 (PL, per row)",
                        xlabel="Combined promoter energy (higher = stronger)"),
}

# Colours follow ELEMENT_COLORS in Model_CorePromoter_energy_vs_conservation.ipynb so
# element identity stays consistent across the repo. The spacer grey is darkened from
# that notebook's #b8b8b8, which disappears at low alpha; ITS is not in that dict and
# takes the seaborn-deep green, PL_combined the seaborn-deep purple.
LIB_COLORS = {
    "UL": "#937860",
    "SL17": "#7f7f7f",
    "DL": "#dd8452",
    "ITS": "#55a868",
    "PL_m35": "#4c72b0",
    "PL_m10": "#c44e52",
    "PL_combined": "#8172b3",
}

# Parameter file backing each library, used only for the cache signature. -35/-10 and
# the combined promoter go through BPM rather than a .pt checkpoint. The spacer branch
# of ElementModelBundle.score() dispatches on sequence length, so a spacer table always
# reaches the checkpoint matching its own length.
_BPM_PARAMS = legacy.SCRIPT_DIR / "BPM" / "Params_Con17.pkl"
LIB_WEIGHTS = {
    "UL": legacy.WEIGHTS_DIR / "weights_UP.pt",
    "SL17": legacy.WEIGHTS_DIR / "weights_Sp17.pt",
    "DL": legacy.WEIGHTS_DIR / "weights_Dis.pt",
    "ITS": legacy.WEIGHTS_DIR / "weights_ITS.pt",
    "PL_m35": _BPM_PARAMS,
    "PL_m10": _BPM_PARAMS,
    "PL_combined": _BPM_PARAMS,
}

print(f"device      : {DEVICE}")
print(f"tables      : {legacy.TABLE_DIR}")
print(f"figures     : {FIG_DIR}")
print(f"cache       : {CACHE_PATH}")
print(f"PL_MIN_ROWS : {PL_MIN_ROWS}   PL_INCLUDE_SPACER: {PL_INCLUDE_SPACER}")
print(f"libraries   : {', '.join(LIBRARIES)}")

In [14]:
# === Boltzmann model shared by both scoring paths ===
# ElementEnergyModel.energy2expression and BPM.score2exp are the SAME two-state
# occupancy with beta pinned to 1/(R*T), just written differently. beta is NOT fitted.
#
# BPM.score2exp writes it as a ratio:
#     B   = exp(-beta*(dG + cE0))
#     GFP = (cmin + cmax*B) / (1 + B)
# Substitute s = B/(1+B), so that 1-s = 1/(1+B):
#     (cmin + cmax*B)/(1+B) = cmin/(1+B) + cmax*B/(1+B)
#                           = cmin*(1-s) + cmax*s
#                           = cmin + (cmax - cmin)*s
# and because our x axis is already higher-is-stronger (e = -dG, since
# ElementModelBundle.score negates the BPM dG):
#     B = exp(-beta*(-e + cE0)) = exp(beta*(e - cE0))  ->  s = sigmoid(beta*(e - cE0))
# which is exactly the form fitted below, with GFP_min=cmin, GFP_max=cmax, e0=cE0.
# Checked numerically against BPM.score2exp over a dG sweep: agreement to ~1e-13.
#
# ElementEnergyModel.energy2expression is the same family under a different
# parameterisation: log(exp(p_min) + exp(p_max)*s) means its plateaus are
# lower = exp(p_min) and upper = exp(p_min) + exp(p_max), not exp(p_max).
#
# Caveat when reading the fitted numbers: BPM's own cE0/cmin/cmax are calibrated on the
# WHOLE promoter dG (score_promoter = m35 + spacer + m10), so a single element's fitted
# e0 is not expected to equal cE0 (7.30), and its upper plateau sits well below cmax
# (1000) because one element on its own never saturates the promoter.
R_GAS = 0.001987   # kcal / mol / K
T_KELVIN = 310.0
BETA = 1.0 / (R_GAS * T_KELVIN)   # 1.6235 mol/kcal


def boltzmann_log10(energy, e0, log10_min, log10_max):
    """Log10 expression as a function of energy, with beta fixed at 1/(R*T).

    Written through expit() rather than exp()/(1+exp()) so it stays finite when the
    optimiser wanders far from the data.
    """
    occupancy = expit(BETA * (energy - e0))
    linear = 10.0**log10_min * (1.0 - occupancy) + 10.0**log10_max * occupancy
    return np.log10(linear)


def fit_boltzmann(x, y):
    """Fit e0 and both plateaus; returns params plus R2 of the fit.

    The plateaus are bounded to the observed expression range. Unbounded, a library
    that never reaches its own lower asymptote (ITS) pushes GFP_min to ~1e-6 for no
    gain in R2, producing a curve whose parameters mean nothing.
    """
    lower = [float(x.min()) - 2.0, float(y.min()) - 1.0, float(y.min())]
    upper = [float(x.max()) + 2.0, float(y.max()), float(y.max()) + 1.0]
    p0 = [float(np.median(x)), float(np.quantile(y, 0.01)), float(np.quantile(y, 0.99))]
    p0 = [float(np.clip(v, lo, hi)) for v, lo, hi in zip(p0, lower, upper)]
    popt, _ = curve_fit(boltzmann_log10, x, y, p0=p0, bounds=(lower, upper), maxfev=40000)
    resid = y - boltzmann_log10(x, *popt)
    r2 = 1.0 - float(np.sum(resid**2)) / float(np.sum((y - y.mean()) ** 2))
    return {"e0": float(popt[0]), "log10_min": float(popt[1]),
            "log10_max": float(popt[2]), "r2": r2}

In [ ]:
# === Load every library, score it, and fit the Boltzmann curve ===
# Expensive cell, deliberately separate from plotting so restyling never rescores.
import BPM.BPM as bpm

BUNDLE = r.ElementModelBundle(DEVICE)


def _combined_promoter_table(df):
    """Whole-promoter BPM energy, one point per row - nothing aggregated.

    This is BPM.score_promoter split into its three additive dG terms and negated onto
    the same higher-is-stronger axis every other panel uses:
        energy = -(dG_m35 + dG_spacer + dG_m10)
    PL stores spacer_length as an int rather than a spacer sequence, so the spacer term
    is read straight from BPM's _dGSP table instead of going through score_spacer().

    The 6-mer score tables are indexed directly rather than called through score_m35()
    / score_m10(), because those return 0.0 for an unknown 6-mer - which would land on
    the energy axis as if it were a real measurement. .map() yields NaN for a miss
    instead, and the isfinite filter below removes it. (Every PL 6-mer is in fact
    covered, so nothing is expected to drop; the guard is there so that a future table
    change is loud rather than silent.)
    """
    missing = {"minus35", "minus10", "spacer_length", "LogGFP"} - set(df.columns)
    if missing:
        raise KeyError(
            f"source='combined' needs the columns {sorted(missing)}, which "
            f"{list(df.columns)} does not provide. Only PL.pkl has this shape."
        )

    dG = (df["minus35"].map(bpm._score35).to_numpy(dtype=float)
          + df["minus10"].map(bpm._score10).to_numpy(dtype=float))
    if PL_INCLUDE_SPACER:
        dG = dG + df["spacer_length"].map(bpm._dGSP).to_numpy(dtype=float)

    out = pd.DataFrame({
        # Categorical, not object: a million rows of 6-char strings would add ~100 MB
        # to the cache, whereas 2,584 / 1,330 distinct values compress to int codes.
        "minus35": pd.Categorical(df["minus35"]),
        "minus10": pd.Categorical(df["minus10"]),
        "spacer_length": df["spacer_length"].to_numpy(),
        "energy": -dG,
        "LogGFP": df["LogGFP"].to_numpy(dtype=float),
    })
    n_raw = len(out)
    out = out[np.isfinite(out["energy"]) & np.isfinite(out["LogGFP"])].reset_index(drop=True)
    if len(out) != n_raw:
        print(f"  dropped {n_raw - len(out)} rows carrying an unscored 6-mer")
    return out


def score_table(filename, element, source="index", min_rows=1):
    """One scored table, ready to plot.

    source="index"    -> pkl index is the sequence and LogGFP is per row.
    source=<column>   -> group by that column and average LogGFP over every row
                         carrying the 6-mer, which is how a single PL element has to be
                         collapsed: a PL row binds a minus35 AND a minus10 to one
                         LogGFP.
    source="combined" -> per-row whole-promoter BPM energy; nothing is aggregated, so
                         every PL row stays a real point.
    """
    df = pd.read_pickle(legacy.TABLE_DIR / filename)

    # Dispatch on source first: the combined mode spans all three core elements, so it
    # has no single element name and must return before the guard below.
    if source == "combined":
        return _combined_promoter_table(df)

    # Fail here rather than as a bare KeyError inside ElementModelBundle.score. The
    # usual cause is a stale LIBRARIES left in the kernel: re-run the setup cell.
    if element not in r.ELEMENT_LENGTHS:
        raise KeyError(
            f"Unknown element {element!r}. Valid elements: {sorted(r.ELEMENT_LENGTHS)}, "
            "plus 'promoter' when source='combined'. Note there is no 'PL' element - "
            "PL is plotted as 'm35'/'m10' with source='minus35'/'minus10', or as the "
            "summed promoter with source='combined'. If this looks wrong, re-run the "
            "setup cell; the kernel may be holding an older LIBRARIES."
        )

    if source == "index":
        seqs = df.index.astype(str).tolist()
        loggfp = df["LogGFP"].to_numpy(dtype=float)
        n_rows = np.ones(len(seqs), dtype=int)
    else:
        grouped = df.groupby(source)["LogGFP"].agg(["mean", "size"])
        if min_rows > 1:
            grouped = grouped[grouped["size"] >= min_rows]
        seqs = grouped.index.astype(str).tolist()
        loggfp = grouped["mean"].to_numpy(dtype=float)
        n_rows = grouped["size"].to_numpy(dtype=int)

    out = pd.DataFrame({
        "sequence": seqs,
        "energy": BUNDLE.score(element, seqs),
        "LogGFP": loggfp,
        "n_rows": n_rows,
    })
    n_raw = len(out)
    out = out[np.isfinite(out["energy"]) & np.isfinite(out["LogGFP"])].reset_index(drop=True)
    if len(out) != n_raw:
        print(f"  dropped {n_raw - len(out)} non-finite rows from {filename}/{source}")
    return out


def fit_stats(df):
    """Correlations plus the fitted Boltzmann parameters for one table."""
    x = df["energy"].to_numpy(dtype=float)
    y = df["LogGFP"].to_numpy(dtype=float)
    row = {
        "n": len(df),
        "energy_min": float(x.min()), "energy_max": float(x.max()),
        "LogGFP_min": float(y.min()), "LogGFP_max": float(y.max()),
        "pearson_r": float(stats.pearsonr(x, y)[0]),
        "spearman_rho": float(stats.spearmanr(x, y)[0]),
    }
    row.update(fit_boltzmann(x, y))
    return row


def _cache_signature():
    """Invalidated when any source table, weight file or PL setting changes."""
    sig = {"PL_MIN_ROWS": PL_MIN_ROWS, "PL_INCLUDE_SPACER": PL_INCLUDE_SPACER}
    for key, meta in LIBRARIES.items():
        sig[f"table:{key}"] = r._file_signature(legacy.TABLE_DIR / meta["file"])
        sig[f"weight:{key}"] = r._file_signature(LIB_WEIGHTS[key])
    return sig


SIGNATURE = _cache_signature()
LIB_DATA = None
LIB_STATS = None

if USE_CACHE and CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as fh:
        cached = pickle.load(fh)
    if cached.get("signature") == SIGNATURE:
        LIB_DATA, LIB_STATS = cached["data"], cached["stats"]
        print(f"Loaded cached energies from {CACHE_PATH}")
    else:
        print("Cache signature is stale (table, weight or PL setting changed); rescoring.")

if LIB_DATA is None:
    LIB_DATA, LIB_STATS = {}, {}
    for key, meta in LIBRARIES.items():
        print(f"scoring {key} ...")
        LIB_DATA[key] = score_table(
            meta["file"], meta["element"], meta["source"], min_rows=PL_MIN_ROWS
        )
        LIB_STATS[key] = fit_stats(LIB_DATA[key])
    with open(CACHE_PATH, "wb") as fh:
        pickle.dump({"signature": SIGNATURE, "data": LIB_DATA, "stats": LIB_STATS}, fh)
    print(f"Wrote cache to {CACHE_PATH}")

summary = pd.DataFrame(LIB_STATS).T
summary["n"] = summary["n"].astype(int)
display(summary.round(4))

In [ ]:
# === Scatter with marginal histograms ===
# Deliberately free of design-bin overlays: no excluded-range shading, no bin-edge
# lines, no energy_fraction secondary axis. The point here is the raw joint
# distribution. English-only labels, per the convention used across this repo.


def scatter_with_marginals(
    df,
    *,
    title,
    color,
    stats_row,
    xlabel="Element energy (higher = stronger)",
    ylabel="Log10 GFP",
    bins=120,
    s=None,
    alpha=None,
    ref_lines=(1.0, 4.0),
    boltzmann=True,
):
    """Joint energy-vs-LogGFP scatter with an energy and a LogGFP marginal histogram.

    ref_lines marks where the element-model training filter (LogGFP > 1 and < 4) would
    have cut, without actually filtering anything.
    """
    x = df["energy"].to_numpy(dtype=float)
    y = df["LogGFP"].to_numpy(dtype=float)
    n = len(df)

    # Markers stay large enough to read as circles rather than pixels; the dense
    # libraries are allowed to saturate. A 1.3k-point PL panel and a 1M-point ITS or
    # PL_combined panel need very different sizes, hence the scaling. Tune the gains
    # and ranges in the setup cell rather than here.
    if s is None:
        s = float(np.clip(POINT_SIZE_GAIN / np.sqrt(n), *POINT_SIZE_RANGE))
    if alpha is None:
        alpha = float(np.clip(POINT_ALPHA_GAIN / n, *POINT_ALPHA_RANGE))

    fig = plt.figure(figsize=(6.2, 6.2), dpi=150)
    gs = fig.add_gridspec(
        2, 2, width_ratios=(4.2, 1.0), height_ratios=(1.0, 4.2), wspace=0.04, hspace=0.04
    )
    ax = fig.add_subplot(gs[1, 0])
    ax_tx = fig.add_subplot(gs[0, 0], sharex=ax)
    ax_ty = fig.add_subplot(gs[1, 1], sharey=ax)

    # rasterized=True collapses the points into one embedded raster layer, so the SVG
    # stays small while axes, ticks and text remain vector.
    ax.scatter(x, y, s=s, alpha=alpha, color=color, marker="o", edgecolors="none",
               linewidths=0, rasterized=True, zorder=3)

    for value in ref_lines:
        if y.min() < value < y.max():
            ax.axhline(value, color="0.65", ls=":", lw=0.9, zorder=2)

    if boltzmann:
        grid = np.linspace(x.min(), x.max(), 400)
        curve = boltzmann_log10(
            grid, stats_row["e0"], stats_row["log10_min"], stats_row["log10_max"]
        )
        # White casing keeps the curve legible where the scatter saturates.
        ax.plot(grid, curve, color="white", lw=3.2, solid_capstyle="round", zorder=4)
        ax.plot(grid, curve, color="black", lw=1.6, zorder=5)

    ax_tx.hist(x, bins=bins, color=color, alpha=0.85)
    ax_ty.hist(y, bins=bins, orientation="horizontal", color=color, alpha=0.85)

    # Pin the limits to the data so the fitted curve cannot expand them. SL16's energy
    # axis, for instance, runs to +3.9 where nothing was ever measured.
    xpad = 0.03 * (x.max() - x.min())
    ypad = 0.03 * (y.max() - y.min())
    ax.set_xlim(x.min() - xpad, x.max() + xpad)
    ax.set_ylim(y.min() - ypad, y.max() + ypad)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax_tx.set_ylabel("Count", fontsize=8)
    ax_ty.set_xlabel("Count", fontsize=8)
    ax_tx.set_title(title, fontsize=11, pad=8)
    ax_tx.tick_params(labelbottom=False)
    ax_ty.tick_params(labelleft=False)
    ax_tx.tick_params(axis="y", labelsize=7)
    ax_ty.tick_params(axis="x", labelsize=7, labelrotation=90)

    for axis in (ax, ax_tx, ax_ty):
        for side in ("top", "right"):
            axis.spines[side].set_visible(False)

    ax.text(
        0.04,
        0.96,
        f"n = {stats_row['n']:,}\n"
        f"Pearson r = {stats_row['pearson_r']:.3f}\n"
        f"Spearman rho = {stats_row['spearman_rho']:.3f}\n"
        f"Boltzmann R2 = {stats_row['r2']:.3f}\n"
        f"e0 = {stats_row['e0']:.2f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.88),
        zorder=6,
    )
    return fig

In [ ]:
# === Render and save one independent figure per library ===
for key, meta in LIBRARIES.items():
    # Only PL_combined overrides the x-axis label; the rest keep the function default.
    extra = {"xlabel": meta["xlabel"]} if "xlabel" in meta else {}
    fig = scatter_with_marginals(
        LIB_DATA[key],
        title=meta["label"],
        color=LIB_COLORS[key],
        stats_row=LIB_STATS[key],
        **extra,
    )
    for ext in ("png", "svg"):
        # dpi at save time sets the resolution of the rasterized point layer.
        fig.savefig(FIG_DIR / f"EnergyVsLogGFP_{key}.{ext}", dpi=300, bbox_inches="tight")
    print(f"{key}: wrote EnergyVsLogGFP_{key}.png / .svg")
    plt.show()

In [ ]:
# === Optional: ad-hoc figure for any table not in the default set ===
# This cell is intentionally left commented out. To use it: uncomment, edit the fields,
# run. Nothing above needs changing, and it does not touch the cache.
# The example below reproduces SL16, which is not in the default set.

# AD_HOC = dict(
#     file="SL16.pkl",              # any pkl in MS2_Data_PyTorch/tables/
#     element="spacer",             # "UP" | "spacer" | "DIS" | "ITS" | "m35" | "m10"
#                                   #   (or "promoter" when source="combined")
#     source="index",               # "index" | a PL column ("minus35" / "minus10")
#                                   #   | "combined" for the summed promoter energy
#     label="Spacer 16 bp (SL16)",
#     color="#7f7f7f",
#     save_as="SL16",               # None to display without writing any file
# )
#
# _df = score_table(AD_HOC["file"], AD_HOC["element"], AD_HOC["source"],
#                   min_rows=PL_MIN_ROWS)
# _stats = fit_stats(_df)
# print(pd.Series(_stats).round(4).to_string())
# _fig = scatter_with_marginals(_df, title=AD_HOC["label"], color=AD_HOC["color"],
#                               stats_row=_stats)
# if AD_HOC["save_as"]:
#     for _ext in ("png", "svg"):
#         _fig.savefig(FIG_DIR / f"EnergyVsLogGFP_{AD_HOC['save_as']}.{_ext}",
#                      dpi=300, bbox_inches="tight")
#     print(f"wrote EnergyVsLogGFP_{AD_HOC['save_as']}.png / .svg")
# plt.show()